In [1]:
import pandas as pd
import numpy as np
import argparse
import sys, os
from pathlib import Path
#import logging
#import line_profiler
sys.path.append('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM')


In [2]:

pdbpath = "/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/1a14.pdb"

In [3]:

from src.preprocess import (strip_pdb,
                            create_dataframe, 
                            extract_chains_chothia,
                            fix_case_if_reused,
                            filter_chart,
                            find_within_radius,
                            closest_atompair,
                            filter_interface,
                            map_atom,
                            flatten_definitions,
                            compute_atom_vectors,
                            pivot_vectors,
                            merge_with_charts,
                            add_orientation_col)

from src.feature_engineering import calc_interface_chartwise, save_triplets_to_npz

from src.config import (charts,
                        #radius_interface,
                        interaction_treshold,
                        ligand_depth,
                        receptor_depth)


import logging
# Configure the root logger
logging.basicConfig(
filename='./app.log',
encoding='utf-8',
filemode='a',
format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
level=logging.INFO
)   




2025-06-03 11:48:02.768840: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-03 11:48:02.843943: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [40]:


def process_direction(ligand_chains: list, receptor_chains: list, complex_data: pd.DataFrame, chart: str, pdb: str) -> list:
    '''
    Takes care of of calculating the chart environments both ways: L to R, R to L
    
    Return:
    List of Tupels: (chart_name, chart_member, interaction_env)
    
    '''
    

    # Ligand and receptor interface data>
    #ligand_data = interface_data[interface_data["chain_id"].isin(ligand_chains)].copy()
    #receptor_data = interface_data[interface_data["chain_id"].isin(receptor_chains)].copy()
    ligand_data,_   = find_within_radius(src = complex_data[complex_data["chain_id"].isin(ligand_chains)],
                                         ref = complex_data[complex_data["chain_id"].isin(receptor_chains)],
                                         radius = ligand_depth)
    
    receptor_data,_ = find_within_radius(src = complex_data[complex_data["chain_id"].isin(receptor_chains)],
                                         ref = ligand_data,
                                         radius = receptor_depth)


    
    
    receptor_data["target atom_id"] = receptor_data.apply(lambda row: map_atom(row['atom_name'], row['res_name']), axis=1)#ToDo: Do not recalculate for every chart

    charts_data = filter_chart(ligand_data, charts[chart])# filter positional and orientational atoms



    #### --------------------------------Add orientation columns, ToDo: Generalize to handle multiple charts
    
    #defs_df = flatten_definitions(charts[chart])
    #with_targets = compute_atom_vectors(charts_data, defs_df)
    #pivot = pivot_vectors(with_targets)
    #charts_data = merge_with_charts(charts_data, pivot, charts[chart])


    return calc_interface_chartwise(
        chart,
        charts_data,
        receptor_data,
        interaction_treshold,
        pdb
    )


In [10]:
sys.argv = ['this_script.py', '--pdbpath', f'{pdbpath}']

#PDB path parser
parser = argparse.ArgumentParser(description="Process a pdb path(str) argument.")

parser.add_argument("-s", "--pdbpath",
                    type=str,
                    required=True,
                    help="Input pdb path to process")


#parser.add_argument("--saveTo",
#                    type=str,
#                    required=True,
#                    help="Optional output path to save the result")

args = parser.parse_args()



In [34]:

#extract chains
try:
    chain_pairs = extract_chains_chothia(args.pdbpath)
    if not chain_pairs:
        raise ValueError(f"No chain pairs returned for {args.pdbpath}")
    ligand_chains, receptor_chains = chain_pairs[0]
    receptor_chains = fix_case_if_reused(ligand_chains, receptor_chains, Path(args.pdbpath).stem) # some chothia pdbs have wrong headers
except Exception as e:
    print(f"[ERROR] Problem processing file: {args.pdbpath}")
    print(f"Reason: {e}")
    exit(1)  # Non-zero exit code so subprocess.run will detect failure


#strip all non-ATOM starting lines from PDB and write to pd.DataFrame
complex_data = create_dataframe(strip_pdb(args.pdbpath))

# Filter interface:
#closest_pair = closest_atompair(complex_data, ligand_chains, receptor_chains)
#center_interface = tuple(closest_pair[0][["x", "y", "z"]])
#interface_data = filter_interface(complex_data, center=center_interface, radius=radius_interface)




In [35]:
complex_data

,record_type,atom_number,atom_name,res_name,chain_id,res_num,x,y,z,occupancy,temp_factor,element
0,ATOM,1,N,ARG,N,82,69.458,78.697,22.692,1.0,42.68,N
1,ATOM,2,CA,ARG,N,82,69.214,77.493,21.841,1.0,39.56,C
2,ATOM,3,C,ARG,N,82,67.900,77.642,21.055,1.0,38.02,C
3,ATOM,4,O,ARG,N,82,67.615,78.714,20.516,1.0,32.05,O
4,ATOM,5,CB,ARG,N,82,69.159,76.215,22.707,1.0,41.31,C
...,...,...,...,...,...,...,...,...,...,...,...,...
5007,ATOM,5120,CA,ALA,L,104,27.501,39.853,77.810,1.0,45.18,C
5008,ATOM,5121,C,ALA,L,104,27.907,38.960,78.987,1.0,52.85,C
5009,ATOM,5122,O,ALA,L,104,29.053,38.446,78.966,1.0,51.11,O
5010,ATOM,5123,CB,ALA,L,104,27.869,39.182,76.473,1.0,46.64,C


In [ ]:


#for chart in ["Guanidino",
#               "Carboxyl",
#                 "Carboxamid",
#                   "Phe_ring",
#                     "Tyr_ring",
#                       "Trp_ring",
#                         "His_ring",
#                           "Pro_ring"]:
#    all_data = []
#
#    for direction in [(ligand_chains, receptor_chains), (receptor_chains, ligand_chains)]:
#        interface_data_chart = process_direction(*direction, complex_data, chart, 
#                                                 Path(args.pdbpath).stem) # these are for logging
#        all_data.extend(interface_data_chart)
#
#
##-----------------------------------------Saving
#    saveToCharts = f"{args.saveTo}/{chart}/raw"
#    os.makedirs(saveToCharts, exist_ok=True)
#    save_triplets_to_npz(all_data, f"{saveToCharts}/{Path(args.pdbpath).stem}")



In [42]:
chart = "Carboxamid"
all_data = []

for direction in [(ligand_chains, receptor_chains), (receptor_chains, ligand_chains)]:
    interface_data_chart = process_direction(*direction, complex_data, chart, 
                                             Path(args.pdbpath).stem) # these are for logging
    all_data.extend(interface_data_chart)

unidentified atom type (hydrogen? belonging to protein) OXT
unidentified atom type (hydrogen? belonging to protein) OXT
unidentified atom type (hydrogen? belonging to protein) OXT


In [43]:
len(all_data[3])
all_data[3][1]

record_type      ATOM
atom_number      1705
atom_name          CD
res_name          GLN
chain_id            N
res_num           296
x              35.128
y              66.589
z              50.611
occupancy         1.0
temp_factor     45.72
element             C
Name: 1704, dtype: object